In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
import os

In [2]:
movies_df  = pd.read_csv('ml-latest-small/movies.csv')
ratings_df = pd.read_csv('ml-latest-small/ratings.csv')
tags_df    = pd.read_csv('ml-latest-small/tags.csv')

print(f'Movies  : {movies_df.shape}')
print(f'Ratings : {ratings_df.shape}')
print(f'Tags    : {tags_df.shape}')

Movies  : (9742, 3)
Ratings : (100836, 4)
Tags    : (3683, 4)


In [3]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# We want to extract year of the movie
movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)', expand=False)
print(f'Missing years: {movies_df["year"].isnull().sum()}')

Missing years: 13


In [5]:
import warnings
warnings.filterwarnings('ignore')
# We added this because there would be a warning if we try to drop on a slice of df

In [6]:
# Dropping movies with missing year
movies_df = movies_df.dropna(subset=['year'])
movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
movies_df = movies_df.dropna(subset=['year'])
print(f'After cleaning: {len(movies_df)}')

After cleaning: 9729


In [7]:
# We want to one-encode the genres of the movies
movies_df['genre_list'] = movies_df['genres'].str.split('|')
mlb = MultiLabelBinarizer()

genre_encoded = pd.DataFrame(
    mlb.fit_transform(movies_df['genre_list']),
    columns=['genre_' + g for g in mlb.classes_],
    index=movies_df.index
)

movies_df = pd.concat([movies_df, genre_encoded], axis=1)
one_hot_genre_cols = ['genre_' + g for g in mlb.classes_]

print(f'Genres: {[g.replace("genre_","") for g in one_hot_genre_cols]}')

Genres: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [8]:
movies_df.head()

,movieId,title,genres,year,genre_list,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,"[Adventure, Animation, Children, Comedy, Fantasy]",0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,"[Adventure, Children, Fantasy]",0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,"[Comedy, Romance]",0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,"[Comedy, Drama, Romance]",0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995,[Comedy],0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
movie_stats = ratings_df.groupby('movieId').agg(
    avg_rating  = ('rating', 'mean'),
    num_ratings = ('rating', 'count'),
    rating_std  = ('rating', 'std'),
).reset_index()
movie_stats['rating_std'] = movie_stats['rating_std'].fillna(0)

# Bayesian average: shrinks low-count movies toward the global mean
# so a movie with 2 five-star ratings doesn't outrank a movie with lower rating but more reviews
C = movie_stats['num_ratings'].mean()
m = movie_stats['avg_rating'].mean()
movie_stats['bayesian_avg'] = (C * m + movie_stats['avg_rating'] * movie_stats['num_ratings']) / (C + movie_stats['num_ratings'])

movies_df = movies_df.merge(movie_stats, on='movieId', how='left')
movies_df[['avg_rating','num_ratings','rating_std','bayesian_avg']] = \
    movies_df[['avg_rating','num_ratings','rating_std','bayesian_avg']].fillna(0)

movies_df[['title','avg_rating','num_ratings','bayesian_avg']].head()

,title,avg_rating,num_ratings,bayesian_avg
0,Toy Story (1995),3.920930,215.0,3.890632
1,Jumanji (1995),3.431818,110.0,3.417227
2,Grumpier Old Men (1995),3.259615,52.0,3.260086
3,Waiting to Exhale (1995),2.357143,7.0,2.897612
4,Father of the Bride Part II (1995),3.071429,49.0,3.104793


In [10]:
# Scaling attributes
scaler = MinMaxScaler()
movies_df['year_norm'] = scaler.fit_transform(movies_df[['year']])
movies_df['num_ratings_norm'] = scaler.fit_transform(movies_df[['num_ratings']])
movies_df['avg_rating_norm'] = movies_df['avg_rating'] / 5.0
movies_df['bayesian_norm'] = movies_df['bayesian_avg'] / 5.0
movies_df['rating_std_norm'] = scaler.fit_transform(movies_df[['rating_std']])

content_feature_cols = one_hot_genre_cols + [
    'year_norm', 'num_ratings_norm', 'avg_rating_norm', 'bayesian_norm', 'rating_std_norm'
]
print(f'Content features: {len(content_feature_cols)}')

Content features: 25


In [12]:
user_cat  = ratings_df['userId'].astype('category')
movie_cat = ratings_df['movieId'].astype('category')

# Making a matrix that describes users rating for each movie
R = csr_matrix(
    (ratings_df['rating'].values,
     (user_cat.cat.codes.values, movie_cat.cat.codes.values)),
    shape=(user_cat.cat.categories.nunique(), movie_cat.cat.categories.nunique())
)

code_to_movieid = dict(enumerate(movie_cat.cat.categories))
print(f'User-movie matrix: {R.shape}  sparsity: {1 - R.nnz/(R.shape[0]*R.shape[1]):.1%}')

# Usign SVD to break into 3 smaller matrix
n_factors = 32
svd = TruncatedSVD(n_components=n_factors, random_state=42)
movie_latent = svd.fit_transform(R.T)
print(f'Explained variance (SVD): {svd.explained_variance_ratio_.sum():.3f}')

# Normalise each movies latent vector
norms = np.linalg.norm(movie_latent, axis=1, keepdims=True) + 1e-8
movie_latent_norm = movie_latent / norms

latent_cols = [f'svd_{i}' for i in range(n_factors)]
latent_df   = pd.DataFrame(movie_latent_norm, columns=latent_cols)
latent_df['movieId'] = [code_to_movieid[i] for i in range(len(latent_df))]
print(f'Latent factor DataFrame: {latent_df.shape}')

User-movie matrix: (610, 9724)  sparsity: 98.3%
Explained variance (SVD): 0.490
Latent factor DataFrame: (9724, 33)


In [13]:
# Combining all features
feature_df = movies_df[['movieId', 'title', 'genres', 'year'] + content_feature_cols].copy()
feature_df = feature_df.merge(latent_df, on='movieId', how='inner')
feature_df = feature_df.dropna().reset_index(drop=True)

all_feature_cols = content_feature_cols + latent_cols
dim = len(all_feature_cols)

print(f'Movies with full feature vectors : {len(feature_df)}')
print(f'Total input features             : {dim}')
print(f'  Content : {len(content_feature_cols)}   SVD latent : {n_factors}')
feature_df.head(3)

Movies with full feature vectors : 9711
Total input features             : 57
  Content : 25   SVD latent : 32


,movieId,title,genres,year,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,...,svd_22,svd_23,svd_24,svd_25,svd_26,svd_27,svd_28,svd_29,svd_30,svd_31
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,0,0,1,1,1,1,...,0.059890,0.047930,0.042637,-0.018202,-0.014047,-0.056507,-0.027629,0.029763,0.130132,-0.143215
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,0,0,1,0,1,0,...,0.065462,0.109900,0.074042,0.012675,0.093461,-0.061287,0.000245,0.017599,0.022820,0.059481
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,0,0,0,0,0,1,...,-0.023063,0.200771,-0.089993,-0.069429,-0.107916,-0.117377,0.106607,0.076988,0.260566,0.012340


Each movie is represented by a 32-dimensional latent vector obtained using Truncated SVD.  
Each `svd_i` column corresponds to one latent factor that captures hidden co-watching patterns between users and movies.

### Example: *Toy Story (1995)*

- `svd_30 = 0.130132`
- `svd_31 = -0.143215`

This means:

- Toy Story has a **positive contribution** to latent factor 30.
- Toy Story has a **negative contribution** to latent factor 31.

In other words, the movie aligns strongly with the behavioral pattern captured by factor 30,  
while it is positioned oppositely along factor 31.

These latent factors do not correspond to explicit genres.  
Instead, they represent hidden viewing patterns learned from user behavior.

In [14]:
X = feature_df[all_feature_cols].values.astype(np.float32)
X_train, X_val = train_test_split(X, test_size=0.1, random_state=42)

In [15]:
X_train.shape

(8739, 57)

In [16]:
X_val.shape

(972, 57)

In [17]:
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

np.save('X_train.npy', X_train)
np.save('X_val.npy',   X_val)

# Movie metadata + full feature table
feature_df[['movieId','title','genres','year']].to_csv('movie_index.csv', index=False)
feature_df.to_csv('movie_features.csv', index=False)

# Column list so other notebooks know the feature order
with open('feature_cols.txt', 'w') as f:
    f.write('\n'.join(all_feature_cols))